In [1]:

import sys
import json
import numpy as np
import os
from PIL import ImageFont
from moviepy import *
from moviepy import vfx
from moviepy.video.VideoClip import TextClip, VideoClip
from moviepy.video.compositing.CompositeVideoClip import CompositeVideoClip

In [ ]:

# FIRST VIDEO (to be worked into thte final montage)

In [ ]:

# WE CAN RUN THIS WHOLE CELL AS A PY FILE (if preferable)
# WE NEED THE TIMELINE AND THE TEXTS (audio_timeline_json, text_json)
# see 'hk_mt_sympoiesis_drift_audio_9.json' for the audio_timeline_json and replace text_json with your own

# ================================================================
#  FONT SETUP
# ================================================================

font_path = os.path.expanduser(YOUR_PATH)

# ================================================================
#  TEXT STYLES
# ================================================================

TEXT_STYLE = {
    "topographical": {
        "font": font_path,
        "font_size": 36,
        "color": "white",
        "stroke_color": "black",
        "stroke_width": 1.5
    },
    "topological": {
        "font": font_path,
        "font_size": 33,
        "color": "cyan",
        "stroke_color": "black",
        "stroke_width": 1.5
    },
    "disruptive": {
        "font": font_path,
        "font_size": 40,
        "color": "red",
        "stroke_color": "black",
        "stroke_width": 2.0
    },
}

INTERFERE = {
    "topographical": "topological",
    "topological": "disruptive",
    "disruptive": "topographical"
}

# ================================================================
#  AUDIO ANALYSIS
# ================================================================

def make_audio_analyzer(audio_clip):
    """
    Returns a function audio_level(t) -> smoothed volume (0..1).
    """
    samples = audio_clip.to_soundarray(fps=44100)
    mono = samples.mean(axis=1)
    win = 2048
    energy = np.sqrt(np.convolve(mono**2, np.ones(win)/win, mode="same"))
    energy /= energy.max() + 1e-6

    times = np.linspace(0, audio_clip.duration, len(energy))

    def audio_level(t):
        if t < 0: return 0
        if t > times[-1]: return 0
        return float(np.interp(t, times, energy) * 0.24)

    return audio_level

# ================================================================
#  AUDIO-REACTIVE BACKGROUND
# ================================================================

def make_audio_reactive_background(size, audio_level, start, end):
    """
    Smooth gradient with subtle audio-reactive shimmer + motion.
    """
    W, H = size

    def make_frame(tlocal):
        t = start + tlocal
        level = audio_level(t)

        base = np.linspace(30, 200, H).reshape(H, 1)
        shift = int(level * 40)
        base = np.roll(base, shift, axis=0)

        R = base
        G = base * (0.6 + 0.4*level)
        B = base * (0.4 + 0.6*level)

        frame = np.stack([R, G, B], axis=2).clip(0,255).astype("uint8")
        return frame

    return VideoClip(make_frame, duration=end-start)

# ================================================================
#  BÉZIER PATH ENGINE (text movement)
# ================================================================

def bezier_path(P0, P1, P2, P3, t):
    return ((1 - t)**3)*P0 + 3*((1-t)**2)*t*P1 + 3*(1-t)*t*t*P2 + t**3*P3

def bezier_position(P0, P1, P2, P3, duration, audio_level, W, H):
    def pos(t):
        u = np.clip(t/duration, 0, 1)

        # audio-reactive speed
        u = min(1, u * (1 + 0.7 * audio_level(t)))

        x, y = bezier_path(np.array(P0), np.array(P1),
                           np.array(P2), np.array(P3), u)

        # -------------------------
        # CLAMP TO SAFE REGION
        # -------------------------
        margin = 50  # prevent text from leaving the frame entirely
        x = np.clip(x, -margin, W - 1)
        y = np.clip(y, -margin, H - 1)

        return (x, y)

    return pos

# ================================================================
#  WORD SHARDS (AUDIO-REACTIVE)
# ================================================================

def generate_shards(text, n=4):
    words = text.split()
    for _ in range(n):
        w = np.random.choice(words)
        if len(w) < 2:
            continue
        high = max(1, len(w)-1)
        start = np.random.randint(0, high)
        end = np.random.randint(start+1, len(w)+1)
        yield w[start:end]

import random

def shard_clip(shard, mode, start, end, screen, audio_level):
    W, H = screen
    st = TEXT_STYLE[mode]
    duration = (end - start) * 5.36

    # --- 1. DYNAMIC CONTENTS ---
    # Add random glitch characters or repeat the shard
    # glitch_chars = ["Δ", "¢", "€", "Ω", "©", "*", "®", "π", "$", "≈", "树, 树", "船, 船", "灯", "谁说亮，says 底，says 沉", "碎影自画", "地面回音", "并非塑性，也非fantastic"]
    glitch_chars = ["树, 树", "船, 船", "灯", "谁说亮，says 底，says 沉", "碎影自画", "地面回音", "并非塑性，也非fantastic"]
    if np.random.random() > 0.36:
        shard = f"{shard}{random.choice(glitch_chars)}"
    
    # --- 2. DYNAMIC SIZE ---
    # Randomize the font size (between 60% and 150% of the base style)
    random_scale = np.random.uniform(0.6, 1.5)
    current_font_size = int(st["font_size"] * random_scale)

    # --- 3. DYNAMIC VELOCITY ---
    x0 = np.random.randint(int(W*0.1), int(W*0.9))
    y0 = np.random.randint(int(H*0.1), int(H*0.9))
    
    # Randomize the "push" strength (velocity)
    velocity_mult = np.random.uniform(30, 120) 

    def pos(t):
        lvl = audio_level(start + t)
        # Higher lvl makes it jitter, velocity_mult makes it travel
        dx = velocity_mult * (lvl + 0.5) 
        dy = velocity_mult * (lvl + 0.5)

        x = x0 + dx * t
        y = y0 + dy * t

        # Use the W-1 and H-1 fix we discussed
        return (np.clip(x, -50, W-1), np.clip(y, -50, H-1))

    # --- 4. CREATE THE CLIP ---
    clip = TextClip(
        text=shard,
        font=st["font"],
        font_size=current_font_size,
        color=st["color"],
        stroke_color=st["stroke_color"],
        stroke_width=st["stroke_width"],
        method="label"
    ).with_start(start).with_duration(duration).with_position(pos)

    # Add a slight random rotation for extra "shattered" feel
    # Note: vfx.Rotate(angle) in MoviePy 2.0+
    clip = clip.with_effects([vfx.Rotate(np.random.randint(-15, 15))])
    
    return clip.with_opacity(0.36)


# ================================================================
#  MAIN TEXT CLIP (audio-reactive Bézier)
# ================================================================

def bezier_text_clip(text, mode, start, end, screen, audio_level):
    W, H = screen
    st = TEXT_STYLE[mode]
    dur = end - start

    P0 = (np.random.randint(0,W), np.random.randint(0,H))
    P1 = (np.random.randint(0,W), np.random.randint(0,H))
    P2 = (np.random.randint(0,W), np.random.randint(0,H))
    P3 = (np.random.randint(0,W), np.random.randint(0,H))

    # path = bezier_position(P0, P1, P2, P3, dur, audio_level)
    path = bezier_position(P0, P1, P2, P3, dur, audio_level, W, H)

    clip = TextClip(
        text=text,
        font=st["font"],
        font_size=st["font_size"],
        color=st["color"],
        stroke_color=st["stroke_color"],
        stroke_width=st["stroke_width"],
        method="label"
    ).with_start(start).with_duration(dur).with_position(path)

    return clip

# ================================================================
#  MODE INTERFERENCE
# ================================================================

def interference_clips(main_seg, text_segments, screen, audio_level):
    t_clips = []
    m = main_seg["mode"]
    alt = INTERFERE[m]

    candidates = [t for t in text_segments if t["mode"] == alt]
    if len(candidates) == 0:
        return t_clips

    # for c in np.random.choice(candidates, min(2, len(candidates)), replace=False):
    # for c in np.random.choice(candidates, min(8, len(candidates)), replace=False):   # for more shards bursting onto screen
    for c in np.random.choice(candidates, min(3, len(candidates)), replace=False):
        tc = bezier_text_clip(
            c["text"], alt,
            c["start"] + np.random.uniform(-0.3,0.6),
            c["end"], screen, audio_level
        ).with_opacity(0.35)
        t_clips.append(tc)

    return t_clips

# ================================================================
#  RUPTURE BURSTS
# ================================================================

def rupture_burst(start, duration, screen, audio_level):
    W, H = screen

    burst = (ColorClip(size=(W,H), color=(255,0,0))
             .with_opacity(0.12)
             .with_effects([
                 vfx.LumContrast(lum=0, contrast=150, contrast_threshold=50)
             ])
             .with_start(start)
             .with_duration(duration))
    return burst

# ================================================================
#  MASTER RENDER FUNCTION
# ================================================================

def render_videopoem(audio_file, audio_timeline_json, text_json, output):
    segments = json.load(open(audio_timeline_json))
    texts = json.load(open(text_json))

    W, H = 1280, 720
    all_layers = []

    audio_clip = AudioFileClip(audio_file)
    audio_level = make_audio_analyzer(audio_clip)

    # BACKGROUNDS
    for seg in segments:
        bg = make_audio_reactive_background((W,H), audio_level, seg["start"], seg["end"])
        bg = bg.with_start(seg["start"]).with_duration(seg["end"] - seg["start"])
        all_layers.append(bg)

    # TEXT + SHARDS + INTERFERENCE + BURSTS
    for t in texts:
        start, end = t["start"], t["end"]
        mode = t["mode"]

        main = bezier_text_clip(t["text"], mode, start, end, (W,H), audio_level)
        all_layers.append(main)

        for shard in generate_shards(t["text"], n=4):
            all_layers.append(shard_clip(shard, mode, start, end, (W,H), audio_level))

        all_layers.extend(interference_clips(t, texts, (W,H), audio_level))

        if mode == "disruptive":
            all_layers.append(rupture_burst(start, end-start, (W,H), audio_level))

    composite = CompositeVideoClip(all_layers, size=(W,H)).with_audio(audio_clip)

    composite.write_videofile(
        output,
        fps=24,
        codec="libx264",
        audio_codec="aac",
        threads=8
    )

    print("✓ Videopoem rendered:", output)

# ================================================================
#  MAIN
# ================================================================

if __name__ == "__main__":
    if len(sys.argv) < 5:
        print("Usage:\n   python render_videopoem.py audio.wav audio.json text.json output.mp4")
        sys.exit(1)

    audio_file = sys.argv[1]
    audio_timeline_json = sys.argv[2]
    text_json = sys.argv[3]
    output = sys.argv[4]

    render_videopoem(audio_file, audio_timeline_json, text_json, output)

In [33]:

import os

#  FONT SETUP
# ================================================================
font_path = os.path.expanduser("YOUR_PATH")

if not os.path.exists(font_path):
    raise FileNotFoundError(f"Double check your path! Font not found at: {font_path}")

# ================================================================
#  TEXT STYLES
# ================================================================

# WE CHANGE THE STYLE A BIT COMPARED TO THE ABOVE

TEXT_STYLE = {
    "topographical": {
        "font": font_path,
        "font_size": 19,
        "color": "white",
        "stroke_color": "black",
        "stroke_width": 1.5
    },
    "topological": {
        "font": font_path,
        "font_size": 33,
        "color": "cyan",
        "stroke_color": "black",
        "stroke_width": 1.5
    },
    "disruptive": {
        "font": font_path,
        "font_size": 40,
        "color": "red",
        "stroke_color": "black",
        "stroke_width": 2.0
    },
}

In [9]:

import moviepy
print(moviepy.__version__)


2.1.2


In [3]:

from moviepy import VideoFileClip, AudioFileClip
from moviepy.video.fx import FadeOut, FadeIn
from moviepy.audio.fx import AudioFadeOut, AudioFadeIn

from moviepy import CompositeVideoClip, VideoFileClip
from moviepy.audio.fx import AudioNormalize

In [ ]:

# SECOND VIDEO

In [26]:

import numpy as np
import matplotlib.pyplot as plt
from moviepy import VideoFileClip, TextClip, ImageClip, CompositeVideoClip


# 1. Helper to generate "Lattice" images
def generate_lattice_image(filename, points_count=50):
    # Generating a simple 2D hexagonal-style lattice (Triangular)
    x = []
    y = []
    rows, cols = int(np.sqrt(points_count)), int(np.sqrt(points_count))
    for i in range(rows):
        for j in range(cols):
            x.append(i + (j % 2) * 0.5)
            y.append(j * np.sqrt(3) / 2)
    
    plt.figure(figsize=(5, 5), facecolor='black')
    plt.scatter(x, y, c='cyan', s=10, alpha=0.8)
    plt.axis('off')
    plt.savefig(filename, bbox_inches='tight', pad_inches=0, facecolor='black')
    plt.close()

# Generate two stages of the lattice
generate_lattice_image("lattice_0.png", points_count=64)
generate_lattice_image("lattice_1.png", points_count=256)
generate_lattice_image("lattice_2.png", points_count=1256)

# 2. Load and Trim Video
main_video = VideoFileClip("margento_live_@_poetry_slam_days_berlin_2009.mp4").subclipped(0, 108)

# main_video = main_video.effects.vfx.fadeout(duration=4)
# main_video = main_video.effects.afx.audio_fadeout(duration=4)

# main_video = main_video.vfx_fadeout(4)
# main_video = main_video.afx_audio_fadeout(4)

# main_video = main_video.effects.fadeout(4)
# main_video = main_video.audio.effects.audio_fadeout(4)

# main_video = fadeout(main_video, 4)
# Note: audio_fadeout needs the audio clip specifically
# main_video.audio = audio_fadeout(main_video.audio, 4)

# Apply video fadeout (visual fade to black)
main_video = main_video.with_effects([FadeOut(duration=4)])

# Apply audio fadeout (sound fades out)
main_video = main_video.with_audio(
    main_video.audio.with_effects([AudioFadeOut(duration=4)]))

st = TEXT_STYLE["topographical"]

# 3. Create Flashes (Text and Lattice)

lattice_flash_0 = ImageClip("lattice_0.png") \
    .with_duration(0.3) \
    .with_start(39.0) \
    .with_position('center')

lattice_flash_01 = ImageClip("lattice_1.png") \
    .with_duration(0.3) \
    .with_start(41.0) \
    .with_position('center')

text_flash_0 = TextClip(
    text="In the beginning was putin—--the Que-\n Bec(n)Oise corrected, poutine—--\n  a tinge, a taint, attained— putin!—α-\n   tenT$ion tones up the Plat(€)-\n    eau which tunes in to the big O st(W)ore Ω-\n     ce[®ul€]an sheen one can sense", 
    font=st["font"],            
    font_size=st["font_size"],  
    color=st["color"],
    stroke_color=st["stroke_color"],
    stroke_width=st["stroke_width"],
    method="label"
    ) \
    .with_duration(0.5) \
    .with_start(48.0) \
    .with_position('center')

text_flash_01 = TextClip(
    text="behind the $¢enes, behind the blind g£oss on st. cat-\n herin(G)’s sky scrapers, the empty lots ec(h)o-\n ing the glisten—--puțin!—--of crashed cans rat-\n  τλing on the silent waves of the big plastic\n   patch drowned out by le lively din on the pro-\    menad€ Du Vi€ux πΟrt—--just one more Péché Mortel on Mar-\n   cello, matter’s cello, la matiére qui dan¢e—--\n  and off to Maisonneuve to listEN to the con-", 
    font=st["font"],            
    font_size=st["font_size"],  
    color=st["color"],
    stroke_color=st["stroke_color"],
    stroke_width=st["stroke_width"],
    method="label"
    ) \
    .with_duration(0.5) \
    .with_start(51.0) \
    .with_position('center')

lattice_flash_1 = ImageClip("lattice_1.png") \
    .with_duration(0.3) \
    .with_start(62.0) \
    .with_position('center')

lattice_flash_11 = ImageClip("lattice_2.png") \
    .with_duration(0.3) \
    .with_start(64.0) \
    .with_position('center')

text_flash_1 = TextClip(
    text="t[®]ainers being loaded we go, cargo\n di la (α)marea largo—--Phu\n  tin tins made en Chine\n   designed by Finns—--tank thin-\n  king sinking in thought ought to think in-\n side the freight box—--in terms", 
    font=st["font"],           
    font_size=st["font_size"],   
    color=st["color"],
    stroke_color=st["stroke_color"],
    stroke_width=st["stroke_width"],
    method="label"
    ) \
    .with_duration(0.5) \
    .with_start(74.0) \
    .with_position('center')

text_flash_11 = TextClip(
    text="of oℕe size fits all: within—in in-\n dustries of faraway con-\n  flicts inflicting extra net-\n   flix flic(k) surv(i)eill(€)ance taxes—the e-\n  vil(£€) axes of ex-comm-\n unicado death excesses of et(h)er-", 
    font=st["font"],           
    font_size=st["font_size"],  
    color=st["color"],
    stroke_color=st["stroke_color"],
    stroke_width=st["stroke_width"],
    method="label"
    ) \
    .with_duration(0.5) \
    .with_start(76.0) \
    .with_position('center')

text_flash_12 = TextClip(
    text="ℕally others... lexIcon$ uncarGo’d in posts where pe(£/$)ts\n are partagéd—--bOrt(h)ă așe, ajunsă—---window dr.\n  ®eS$ing screen caressing & grAnd g£am(β) en mass(€)-\n    acre aching not just hot to t®ot lot t(w)o πlot gro(w)-\n   apã in graVitas gℝaves—--Никто не ever comes\n  back но angst ложится в postель—--no one ret-\n urns but angst—--logically, I$—--lodged in bed, b-\nox-spring, happy sleeping food, can(n)on[t]Ic fodder", 
    font=st["font"],            
    font_size=st["font_size"],   
    color=st["color"],
    stroke_color=st["stroke_color"],
    stroke_width=st["stroke_width"],
    method="label"
    ) \
    .with_duration(0.5) \
    .with_start(100.0) \
    .with_position('center')

# 4. Composite and Normalize Audio
final_video = CompositeVideoClip([main_video, lattice_flash_0, lattice_flash_01, text_flash_0, text_flash_01, lattice_flash_1, lattice_flash_11, text_flash_1, text_flash_11, text_flash_12])

# Normalize audio volume
# final_video.audio = final_video.audio.fx(audio_normalize)

# final_video.write_videofile("margento_sympoiesis_61.mp4", codec="libx264", audio_codec="aac")
# final_video = final_video.fx(afx.audio_normalize)

# Apply audio normalization - new syntax (that is why we checked the MoviePy version upstream)
if final_video.audio:
    final_video = final_video.with_audio(
        final_video.audio.with_effects([AudioNormalize()])
    )

# Write the final video
# final_video.write_videofile("margento_sympoiesis_6_1.mp4")
# final_video.write_videofile("margento_sympoiesis_6_11.mp4")
# final_video.write_videofile("margento_sympoiesis_6_12.mp4")
# final_video.write_videofile("margento_sympoiesis_6_13.mp4")

final_video.write_videofile("margento_sympoiesis_6_131.mp4")

MoviePy - Building video margento_sympoiesis_6_131.mp4.
MoviePy - Writing audio in margento_sympoiesis_6_131TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video margento_sympoiesis_6_131.mp4



MoviePy - Done !
MoviePy - video ready margento_sympoiesis_6_131.mp4


In [ ]:

# THIRD VIDEO

In [8]:

import numpy as np
import matplotlib.pyplot as plt

def generate_lattice_image(filename, points_count=50):
    # Generating a simple 2D hexagonal-style lattice (Triangular)
    x = []
    y = []
    rows, cols = int(np.sqrt(points_count)), int(np.sqrt(points_count))
    for i in range(rows):
        for j in range(cols):
            x.append(i + (j % 2) * 0.5)
            y.append(j * np.sqrt(3) / 2)
    
    plt.figure(figsize=(5, 5), facecolor='black')
    plt.scatter(x, y, c='cyan', s=10, alpha=0.8)
    plt.axis('off')
    plt.savefig(filename, bbox_inches='tight', pad_inches=0, facecolor='black')
    plt.close()

# Generate two stages of the lattice
# generate_lattice_image("lattice_0.png", points_count=64)
# generate_lattice_image("lattice_1.png", points_count=256)
# generate_lattice_image("lattice_2.png", points_count=1256)

generate_lattice_image("lattice_2.png", points_count=1256)
generate_lattice_image("lattice_3.png", points_count=2324)

In [27]:

from moviepy import AudioFileClip

# Load the audio file and check its properties
# audio_clip = AudioFileClip("0348_floating_dock_river_saint_laurent_29-05-15_clean.wav")
audio_clip = AudioFileClip("149748__rtb45__yuen-po-st-bird-garden-hong-kong_clean.wav")

print(f"Duration: {audio_clip.duration} seconds")
print(f"FPS/Sample rate: {audio_clip.fps}")
print(f"Audio exists: {audio_clip is not None}")

# Close the clip to free resources
audio_clip.close()

Duration: 136.3 seconds
FPS/Sample rate: 44100
Audio exists: True


In [ ]:

v0 = AudioFileClip("0348_floating_dock_river_saint_laurent_29-05-15_clean.wav").subclipped(0, 18)
v1 = AudioFileClip("149748__rtb45__yuen-po-st-bird-garden-hong-kong_clean.wav").subclipped(0, 28)

# v1 = v1.with_effects([FadeOut(duration=10)])

# v1 = v1.with_effects([
    # AudioNormalize(),
    # AudioFadeOut(duration=10)
#])

In [29]:

# W, H = 1280, 720
audio_finale = CompositeAudioClip([v0, v1.with_start(v0.duration)])

# Create a silent video clip with the same duration as the audio
# video_finale = ColorClip(size=(1920, 1080), color=(0, 0, 0), duration=final_audio.duration)
video_finale = ColorClip(size=(1280, 720), color=(0, 0, 0), duration=audio_finale.duration)

# Attach audio to video
main_finale = video_finale.with_audio(audio_finale)

main_finale.write_videofile("main_finale.mp4", fps=24)

MoviePy - Building video main_finale.mp4.
MoviePy - Writing audio in main_finaleTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video main_finale.mp4



MoviePy - Done !
MoviePy - video ready main_finale.mp4


In [31]:

# Apply video fadeout (visual fade to black)
main_finale = main_finale.with_effects([FadeOut(duration=10)])

# Apply audio fadeout (sound fades out)
main_finale = main_finale.with_audio(
    main_finale.audio.with_effects([AudioFadeOut(duration=10)]))

main_finale.write_videofile("main_finale_1.mp4", fps=24)

MoviePy - Building video main_finale_1.mp4.
MoviePy - Writing audio in main_finale_1TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video main_finale_1.mp4



MoviePy - Done !
MoviePy - video ready main_finale_1.mp4


In [39]:

st = TEXT_STYLE["topographical"]

# generate_lattice_image("lattice_4.png", points_count=3324)

lattice_flash_2 = ImageClip("lattice_2.png") \
    .resize(main_finale.size) \
    .with_duration(0.3) \
    .with_start(9.0) \
    .with_position('center')

lattice_flash_21 = ImageClip("lattice_3.png") \
    .resize(main_finale.size) \
    .with_duration(0.3) \
    .with_start(11.0) \
    .with_position('center')

text_flash_2 = TextClip(
    text="for the go(o)d(d) father OPe®rating the u-\n crane, jaws-of-life getting you h-\n  OM(μ)€ in a boXXX—a cubicle in the off(ing)-\n   I¢e meTro’s tℝue tRoug(€)h Chinat\n  own noOd(£)e place one would love tHeir lie-\nΦe to be bilingu€(f)facted up", 
    font=st["font"],
    font_size=st["font_size"],
    color=st["color"],
    stroke_color=st["stroke_color"],
    stroke_width=st["stroke_width"],
    method="label"
    ) \
    .with_duration(0.5) \
    .with_start(16.0) \
    .with_position('center')

text_flash_21 = TextClip(
    text="at—on\nward is up\nwar\nd de mo\nun\nt\n\nro ya al£\nof dis o\nbed i.e. ℕ\n¢€ of man\nSion son bi\nshop st\nre et st ant\noIne—the blaℕc\n\nk BOw\nXo\nf dis\no be\ndie\n($¢i) €nce", 
    font=st["font"],
    font_size=st["font_size"],
    color=st["color"],
    stroke_color=st["stroke_color"],
    stroke_width=st["stroke_width"],
    method="label"
    ) \
    .with_duration(0.5) \
    .with_start(17.0) \
    .with_position('center')

lattice_flash_22 = ImageClip("lattice_3.png") \
    .resize(main_finale.size) \
    .with_duration(0.3) \
    .with_start(22.0) \
    .with_position('center')

lattice_flash_23 = ImageClip("lattice_4.png") \
    .resize(main_finale.size) \
    .with_duration(24.3) \
    .with_start(11.0) \
    .with_position('center')


finale_video = CompositeVideoClip([main_finale, lattice_flash_2, lattice_flash_21, text_flash_2, text_flash_21, lattice_flash_22, lattice_flash_23])

# NO NEED FOR THIS HERE, THEY HAVE BEEN NORMALIZED (SEE ABOVE)
# if final_video.audio:
    # final_video = final_video.with_audio(
        # final_video.audio.with_effects([AudioNormalize()])
    #)


finale_video.write_videofile("margento_sympoiesis_6_134.mp4", fps=24)


AttributeError: 'ImageClip' object has no attribute 'resize'

In [40]:


# Get your video dimensions from main_finale
video_width, video_height = main_finale.size

# Scale images to fit screen
lattice_flash_2 = (ImageClip("lattice_2.png")
    .resized(new_size=(video_width, video_height))  # Scale to full screen
    .with_duration(0.3)
    .with_start(9.0)
    .with_position('center'))

lattice_flash_21 = (ImageClip("lattice_3.png")
    .resized(new_size=(video_width, video_height))
    .with_duration(0.3)
    .with_start(11.0)
    .with_position('center'))

lattice_flash_22 = (ImageClip("lattice_3.png")
    .resized(new_size=(video_width, video_height))
    .with_duration(0.3)
    .with_start(22.0)
    .with_position('center'))

lattice_flash_23 = (ImageClip("lattice_4.png")
    .resized(new_size=(video_width, video_height))
    .with_duration(0.3)
    .with_start(24.0)
    .with_position('center'))

text_flash_2 = TextClip(
    text="for the go(o)d(d) father OPe®rating the u-\n crane, jaws-of-life getting you h-\n  OM(μ)€ in a boXXX—a cubicle in the off(ing)-\n   I¢e meTro’s tℝue tRoug(€)h Chinat\n  own noOd(£)e place one would love tHeir lie-\nΦe to be bilingu€(f)facted up", 
    font=st["font"],
    font_size=st["font_size"],
    color=st["color"],
    stroke_color=st["stroke_color"],
    stroke_width=st["stroke_width"],
    method="label"
    ) \
    .with_duration(0.5) \
    .with_start(16.0) \
    .with_position('center')

text_flash_21 = TextClip(
    text="at—on\nward is up\nwar\nd de mo\nun\nt\n\nro ya al£\nof dis o\nbed i.e. ℕ\n¢€ of man\nSion son bi\nshop st\nre et st ant\noIne—the blaℕc\n\nk BOw\nXo\nf dis\no be\ndie\n($¢i) €nce", 
    font=st["font"],
    font_size=st["font_size"],
    color=st["color"],
    stroke_color=st["stroke_color"],
    stroke_width=st["stroke_width"],
    method="label"
    ) \
    .with_duration(0.5) \
    .with_start(17.0) \
    .with_position('center')

finale_video = CompositeVideoClip([main_finale, lattice_flash_2, lattice_flash_21, text_flash_2, text_flash_21, lattice_flash_22, lattice_flash_23])

# NO NEED FOR THIS HERE, THEY HAVE BEEN NORMALIZED (SEE ABOVE)
# if final_video.audio:
    # final_video = final_video.with_audio(
        # final_video.audio.with_effects([AudioNormalize()])
    #)

finale_video.write_videofile("margento_sympoiesis_6_135.mp4", fps=24)

MoviePy - Building video margento_sympoiesis_6_135.mp4.
MoviePy - Writing audio in margento_sympoiesis_6_135TEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video margento_sympoiesis_6_135.mp4



MoviePy - Done !
MoviePy - video ready margento_sympoiesis_6_135.mp4


In [ ]:
# CREDITS

In [47]:


video_width, video_height = main_finale.size
# ============================================
# Define timing (in seconds)
# ============================================
title_duration = 3.0      # How long the title stays on screen
rest_duration = 2.0       # Brief pause between title and authors
authors_duration = 3.0    # How long the authors stay on screen
fade_duration = 1.0       # Duration of fade in/out transitions

# credits_start = main_finale.duration  # Or specify a specific time
credits_start = 36.0

# ============================================
# Create title screen
# ============================================
title_text = TextClip(
    text="LET THE NOISE IN\nSYMPOIESIS",
    font=st["font"],
    font_size=42,
    color="white",
    stroke_color="black",
    stroke_width=2,
    method="label"
).with_position('center')


# title_background = ColorClip(
    # size=(video_width, video_height),
    # color=(0, 0, 0)  # Black background
# ).with_duration(title_duration)

# Composite title with background
title_screen = CompositeVideoClip([
    # title_background,
    title_text.with_duration(title_duration)
])

# Apply fade effects
title_screen = title_screen.with_effects([
    FadeIn(fade_duration),
    FadeOut(fade_duration)
])

# ============================================
# Create authors screen
# ============================================
# Option A: Single text block with multiple lines
authors_text = TextClip(
    text="by\nMARGENTO",
    font=st["font"],
    font_size=43,
    color="white",
    stroke_color="black",
    stroke_width=1,
    method="label",
    #interline=10  # Adjust line spacing (negative = tighter)
).with_position('center')

authors_screen = CompositeVideoClip([
    # title_background,
    authors_text.with_duration(authors_duration)
])

# Apply fade effects
authors_screen = authors_screen.with_effects([
    FadeIn(fade_duration),
    FadeOut(fade_duration)
])

# ============================================
# Create rest/blank screen between title and authors
# ============================================
rest_screen = ColorClip(
    size=(video_width, video_height),
    color=(0, 0, 0)
).with_duration(rest_duration)

# Apply subtle fade if desired
rest_screen = rest_screen.with_effects([
    FadeIn(fade_duration * 0.5),
    FadeOut(fade_duration * 0.5)
])

# ============================================
# Combine everything
# ============================================
credits_sequence = CompositeVideoClip([
    title_screen.with_start(credits_start),
    rest_screen.with_start(credits_start + title_duration),
    authors_screen.with_start(credits_start + title_duration + rest_duration)
])

# ============================================
# Add credits to your main video
# ============================================
finale_video_with_credits = CompositeVideoClip([
    finale_video,
    credits_sequence
])

# Export
# finale_video_with_credits.write_videofile("margento_6_134_sympoiesis_with_credits.mp4", fps=24)
# finale_video_with_credits.write_videofile("margento_6_135_sympoiesis_with_credits.mp4", fps=24)

finale_video_with_credits.write_videofile("margento_6_136_sympoiesis_with_credits.mp4", fps=24)

MoviePy - Building video margento_6_136_sympoiesis_with_credits.mp4.
MoviePy - Writing audio in margento_6_136_sympoiesis_with_creditsTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
MoviePy - Writing video margento_6_136_sympoiesis_with_credits.mp4



MoviePy - Done !
MoviePy - video ready margento_6_136_sympoiesis_with_credits.mp4


In [ ]:

# FINAL MONTAGE


In [ ]:

video0 = VideoFileClip("margento_sympoiesis_6.mp4").subclipped(0, 240)
video1 = VideoFileClip("margento_sympoiesis_6_131.mp4")
video2 = finale_video_with_credits # "margento_6_136_sympoiesis_with_credits.mp4"

final_sympoiesis = CompositeVideoClip([video0, video1, video2])

if final_sympoiesis.audio:
    final_sympoiesis = final_sympoiesis.with_audio(
        final_sympoiesis.audio.with_effects([AudioNormalize()])
    )

final_sympoiesis.write_videofile("margento_sympoiesis_let_the_noise_in.mp4", fps=24)

In [ ]:

# INSERTIONS

In [53]:

ins_video = VideoFileClip("margento_live_@_poetry_slam_days_berlin_2009.mp4").subclipped(100, 104)

In [55]:

# Define insertion time (in seconds) into the finale video
insert_time = 173.0 

# Create final video with excerpt
final_sympoiesis_with_excerpt = CompositeVideoClip([
    final_sympoiesis,
    ins_video.with_start(insert_time)
])

if final_sympoiesis_with_excerpt.audio:
    final_sympoiesis_with_excerpt = final_sympoiesis_with_excerpt.with_audio(
        final_sympoiesis_with_excerpt.audio.with_effects([AudioNormalize()])
    )

video_width, video_height = final_sympoiesis.size

generate_lattice_image("lattice_5.png", points_count=4324)

# Scale images to fit screen
lattice_flash_3 = (ImageClip("lattice_4.png")
    .resized(new_size=(video_width, video_height))  # Scale to full screen
    .with_duration(0.3)
    .with_start(204.0)
    .with_position('center'))

lattice_flash_31 = (ImageClip("lattice_5.png")
    .resized(new_size=(video_width, video_height))
    .with_duration(0.3)
    .with_start(206.0)
    .with_position('center'))


In [ ]:

# THE FINAL VIDEO

In [56]:

sympoiesis_final = CompositeVideoClip([final_sympoiesis_with_excerpt, lattice_flash_3, lattice_flash_31])

sympoiesis_final.write_videofile("margento_sympoiesis.mp4", fps=24)

frame_index:   1%|▏              | 124/8796 [7:09:55<11:27, 12.62it/s, now=None]

MoviePy - Building video margento_sympoiesis.mp4.
MoviePy - Writing audio in margento_sympoiesisTEMP_MPY_wvf_snd.mp3



frame_index:   1%|▏              | 124/8796 [7:09:57<11:27, 12.62it/s, now=None]

MoviePy - Done.
MoviePy - Writing video margento_sympoiesis.mp4




frame_index:   1%|▏              | 124/8796 [7:18:11<11:27, 12.62it/s, now=None]

MoviePy - Done !
MoviePy - video ready margento_sympoiesis.mp4
